In [11]:
def resolver_sistema_no_lineal():
    # 1. Definir las variables simbólicas (G, I, B donde B representa las células beta)
    G, I, B = sp.symbols('G I B', real=True)

    # 2. Definir las constantes usando los valores de la tabla
    R_0 = 864
    G_e = 140
    E_GO = 1.44
    S_I = 0.72
    sigma = 43.2
    alpha = 20000
    rho = 0.41
    k = 432
    d_0 = 0.06
    r_1 = 0.84e-3   # Equivale a 0.84 * 10^-3
    r_2 = 0.24e-5   # Equivale a 0.24 * 10^-5

    # 3. Formar las ecuaciones igualadas a cero
    # eq1: R_0 + G_e - (E_GO + S_I*I)*G = 0
    eq1 = sp.Eq(R_0 + G_e - (E_GO + S_I * I) * G, 0)
    
    # eq2: (B * sigma * G^2) / (alpha + G^2) - (rho - k)*I = 0
    eq2 = sp.Eq((B * sigma * G*2) / (alpha + G*2) - (rho + k) * I, 0)
    
    # eq3: (-d_0 + r_1*G - r_2*G^2)*B = 0
    eq3 = sp.Eq((-d_0 + r_1 * G - r_2 * G**2) * B, 0)

    # 4. Resolver el sistema
    print("Calculando soluciones...\n")
    soluciones = sp.nonlinsolve([eq1, eq2, eq3], [G, I, B])

    print("Las soluciones encontradas son:")
    
    if not soluciones:
        print("El sistema no tiene solución real o compleja.")
    else:
        for i, sol in enumerate(soluciones):
            # sol es una tupla con el orden (G, I, B)
            G_sol = sol[0]
            I_sol = sol[1]
            B_sol = sol[2]
            
            # Usamos evalf() para forzar el resultado a decimales legibles
            print(f"\nSolución {i + 1}:")
            print(f"  Glucosa (G) = {G_sol.evalf(5)}")
            print(f"  Insulina (I)= {I_sol.evalf(5)}")
            print(f"  Células (B) = {B_sol.evalf(5)}")

resolver_sistema_no_lineal()

Calculando soluciones...

Las soluciones encontradas son:

Solución 1:
  Glucosa (G) = -10000
  Insulina (I)= -2.1394
  Células (B) = 0

Solución 2:
  Glucosa (G) = 697.22
  Insulina (I)= 0
  Células (B) = 0


In [ ]:
import sympy as sp

def clasificar_puntos_criticos(ecuaciones, variables, puntos_criticos):
    """
    Analiza y clasifica una lista de puntos críticos de un sistema dinámico 
    usando el método de Hartman-Grobman.
    """
    print("=== ANÁLISIS DE PUNTOS CRÍTICOS (HARTMAN-GROBMAN) ===\n")
    
    # Paso 2: Obtener la Matriz Jacobiana de forma simbólica
    sistema_matriz = sp.Matrix(ecuaciones)
    J = sistema_matriz.jacobian(variables)
    
    print("1. Matriz Jacobiana General:")
    sp.pprint(J)
    print("\n" + "="*50 + "\n")
    
    # Iterar sobre cada punto crítico dado
    for i, punto in enumerate(puntos_criticos, 1):
        print(f"--- Evaluando Punto Crítico {i}: {punto} ---")
        
        # Crear un diccionario para sustituir las variables por los valores del punto
        subs_dict = dict(zip(variables, punto))
        
        # Evaluar la matriz Jacobiana en el punto crítico
        J_eval = J.subs(subs_dict)
        print("Matriz Jacobiana evaluada:")
        sp.pprint(J_eval)
        
        # Paso 3: Calcular los eigenvalores (valores propios)
        # eigenvals() devuelve un diccionario {eigenvalor: multiplicidad}
        eigenvalores_dict = J_eval.eigenvals()
        eigenvalores = list(eigenvalores_dict.keys())
        
        print("\nEigenvalores:")
        for ev in eigenvalores:
            print(f"  λ = {ev}")
        
        # Extraer las partes reales e imaginarias
        partes_reales = [sp.re(ev) for ev in eigenvalores]
        partes_imag = [sp.im(ev) for ev in eigenvalores]
        
        # Paso 4: Verificar hiperbolicidad
        # Es hiperbólico si NINGUNA parte real es exactamente cero
        es_hiperbolico = all(r != 0 for r in partes_reales)
        
        # Paso 5: Clasificación
        print("\nDiagnóstico:")
        if not es_hiperbolico:
            print("  ⚠️ NO HIPERBÓLICO.")
            print("  Al menos un eigenvalor tiene parte real cero.")
            print("  El Teorema de Hartman-Grobman NO ES CONCLUYENTE aquí.")
        else:
            print("  ✓ Punto Hiperbólico. Procediendo a clasificar...")
            
            tiene_imaginarios = any(im != 0 for im in partes_imag)
            
            if all(r < 0 for r in partes_reales):
                if tiene_imaginarios:
                    print("  CLASIFICACIÓN: Foco / Espiral Atractor (Sumidero Estable)")
                else:
                    print("  CLASIFICACIÓN: Nodo Atractor (Sumidero Estable)")
                    
            elif all(r > 0 for r in partes_reales):
                if tiene_imaginarios:
                    print("  CLASIFICACIÓN: Foco / Espiral Repulsor (Fuente Inestable)")
                else:
                    print("  CLASIFICACIÓN: Nodo Repulsor (Fuente Inestable)")
                    
            else:
                print("  CLASIFICACIÓN: Punto de Silla (Saddle Inestable)")
                
        print("="*50 + "\n")

# ==========================================
# EJEMPLO DE USO
# ==========================================
if __name__ == "__main__":
    # 1. Definir las variables simbólicas (G, I, B donde B representa las células beta)
    G, I, B = sp.symbols('G I B', real=True)

    # 2. Definir las constantes usando los valores de la tabla
    R_0 = 864
    G_e = 140
    E_GO = 1.44
    S_I = 0.72
    sigma = 43.2
    alpha = 20000
    rho = 0.41
    k = 432
    d_0 = 0.06
    r_1 = 0.84e-3   # Equivale a 0.84 * 10^-3
    r_2 = 0.24e-5   # Equivale a 0.24 * 10^-5

    variables = [G, I, B]
    
    # Definir el sistema de ecuaciones no lineales
    # Ejemplo: dx/dt = y, dy/dt = x - x^3 - y
    ecuacion_1 = R_0 + G_e - (E_GO + S_I * I) * G
    ecuacion_2 = (B * sigma * G*2) / (alpha + G*2) - (rho + k) * I
    ecuacion_3 = (-d_0 + r_1 * G - r_2 * G**2) * B
    sistema = [ecuacion_1, ecuacion_2, ecuacion_3]
    
    # Definir los puntos críticos que ya conoces
    # En este sistema, al igualar a cero, los puntos son (-1, 0, 0), (0, 0, 0) y (1, 0, 0)
    puntos = [(697.22, 0, 0), (0, 0, 0), (1, 0, 0)]
    
    # Ejecutar la función
    clasificar_puntos_criticos(sistema, variables, puntos)

=== ANÁLISIS DE PUNTOS CRÍTICOS (HARTMAN-GROBMAN) ===

1. Matriz Jacobiana General:
⎡        -0.72⋅I - 1.44          -0.72⋅G                0               ⎤
⎢                                                                       ⎥
⎢    172.8⋅B⋅G        86.4⋅B                         86.4⋅G             ⎥
⎢- ────────────── + ───────────  -432.41           ───────────          ⎥
⎢               2   2⋅G + 20000                    2⋅G + 20000          ⎥
⎢  (2⋅G + 20000)                                                        ⎥
⎢                                                                       ⎥
⎢                                                   2                   ⎥
⎣    B⋅(0.00084 - 4.8e-6⋅G)         0     - 2.4e-6⋅G  + 0.00084⋅G - 0.06⎦


--- Evaluando Punto Crítico 1: (-1, 0, 0) ---
Matriz Jacobiana evaluada:
⎡-1.44   0.72             0          ⎤
⎢                                    ⎥
⎢  0    -432.41  -0.00432043204320432⎥
⎢                                    ⎥
⎣  0       0       